In [1]:
import pulp
import numpy as np
import pandas as pd

In [2]:
problem = pulp.LpProblem('SLE', pulp.LpMaximize)

x = pulp.LpVariable('x', cat='Continuous')
y = pulp.LpVariable('y', cat='Continuous')

problem += 120 * x + 150 * y == 1440
problem += x + y == 10

status = problem.solve()

print(f'Status: {pulp.LpStatus[status]}')
print(f'x = {x.varValue}', f'y = {y.varValue}')





Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/conda/lib/python3.11/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/a5113aa5f05a47628105921cb3c1fca5-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /tmp/a5113aa5f05a47628105921cb3c1fca5-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 7 COLUMNS
At line 13 RHS
At line 16 BOUNDS
At line 20 ENDATA
Problem MODEL has 2 rows, 3 columns and 4 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 0 (-2) rows, 0 (-3) columns and 0 (-4) elements
Empty problem - 0 rows, 0 columns and 0 elements
Optimal - objective value -0
After Postsolve, objective 0, infeasibilities - dual 0 (0), primal 0 (0)
Optimal objective 0 - 0 iterations time 0.002, Presolve 0.00
Option for printingOptions changed from normal to all
Total time (CPU seconds):       0.00   (Wallclock seconds):       0.00

Status: O

In [3]:
problem = pulp.LpProblem('LP', pulp.LpMaximize)

x = pulp.LpVariable('x', cat='Continuous')
y = pulp.LpVariable('y', cat='Continuous')

problem += 1 * x + 3 * y <= 30
problem += 2 * x + 1 * y <= 40
problem += x >= 0
problem += y >= 0
problem += x + 2 * y

status = problem.solve()
print(f'Status: {pulp.LpStatus[status]}')
print(f'x = {x.varValue}', f'y = {y.varValue}', 'obj=', pulp.value(problem.objective)) 



Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/conda/lib/python3.11/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/268cca873ccb4e97a0ddb9898ad3e0d7-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /tmp/268cca873ccb4e97a0ddb9898ad3e0d7-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 9 COLUMNS
At line 18 RHS
At line 23 BOUNDS
At line 26 ENDATA
Problem MODEL has 4 rows, 2 columns and 6 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 2 (-2) rows, 2 (0) columns and 4 (-2) elements
0  Obj -0 Dual inf 2.999998 (2)
2  Obj 26
Optimal - objective value 26
After Postsolve, objective 26, infeasibilities - dual 0 (0), primal 0 (0)
Optimal objective 26 - 2 iterations time 0.002, Presolve 0.00
Option for printingOptions changed from normal to all
Total time (CPU seconds):       0.00   (Wallclock seconds):       0.00

Status: Optimal

### 2.3

In [4]:
stock_df =pd.read_csv('/home/jovyan/data/2.tutorial/stocks.csv')
stock_df.head()

,m,stock
0,m1,35
1,m2,22
2,m3,27


In [5]:
require_df = pd.read_csv('/home/jovyan/data/2.tutorial/requires.csv')
require_df.head()

,p,m,require
0,p1,m1,2
1,p1,m2,0
2,p1,m3,1
3,p2,m1,3
4,p2,m2,2


In [6]:
gain_df = pd.read_csv('/home/jovyan/data/2.tutorial/gains.csv')
gain_df.head()

,p,gain
0,p1,3
1,p2,4
2,p3,4
3,p4,5


In [7]:
P = gain_df['p'].tolist()
print(P)

['p1', 'p2', 'p3', 'p4']


In [8]:
M = stock_df['m'].tolist()
print(M)

['m1', 'm2', 'm3']


In [9]:
stock = {row.m:row.stock for row in stock_df.itertuples()}
print(stock)

{'m1': 35, 'm2': 22, 'm3': 27}


In [10]:
stock_dict = dict(zip(stock_df['m'], stock_df['stock']))
print(stock_dict)

{'m1': 35, 'm2': 22, 'm3': 27}


In [11]:
stock = stock_df.set_index('m').to_dict()['stock']
print(stock)

{'m1': 35, 'm2': 22, 'm3': 27}


In [12]:
require = {(row.p, row.m): row.require for row in require_df.itertuples()}
print(require)

{('p1', 'm1'): 2, ('p1', 'm2'): 0, ('p1', 'm3'): 1, ('p2', 'm1'): 3, ('p2', 'm2'): 2, ('p2', 'm3'): 0, ('p3', 'm1'): 0, ('p3', 'm2'): 2, ('p3', 'm3'): 2, ('p4', 'm1'): 2, ('p4', 'm2'): 2, ('p4', 'm3'): 2}


In [13]:
gain = {row.p: row.gain for row in gain_df.itertuples()}
print(gain)

{'p1': 3, 'p2': 4, 'p3': 4, 'p4': 5}


In [14]:
problem = pulp.LpProblem('LP2', pulp.LpMaximize)

In [15]:
x = pulp.LpVariable.dicts('x', P, cat='Integer')

In [16]:
#x = {}
#for p in P:
#    x[p] = pulp.LpVariable(f'x_{p}', cat='Continuous')

In [17]:
#x = {p: pulp.LpVariable(f'x_{p}', cat='Continuous') for p in P}

In [18]:
for p in P:
    problem += x[p] >= 0

In [19]:
for m in M:
    problem += pulp.lpSum([require[p, m] * x[p] for p in P]) <= stock[m]

In [20]:
problem += pulp.lpSum([gain[p] * x[p] for p in P])

In [21]:
status = problem.solve()
print(f'Status: {pulp.LpStatus[status]}')

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/conda/lib/python3.11/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/031a7c2dc95345ed9327df96489b2563-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /tmp/031a7c2dc95345ed9327df96489b2563-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 12 COLUMNS
At line 38 RHS
At line 46 BOUNDS
At line 51 ENDATA
Problem MODEL has 7 rows, 4 columns and 13 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 80.4286 - 0.00 seconds
Cgl0004I processed model has 3 rows, 4 columns (4 integer (0 of which binary)) and 9 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0012I Integer solution of -76 found by DiveCoefficient after 0 iterations and 0 nodes (0.00 seconds)
Cbc0038I Full problem 3 rows 4 columns, reduced to 3 rows 3 columns
Cbc0012I Integer solution of -79 fo

In [22]:
for p in P:
    print(p, x[p].value())
print('obj=', problem.objective.value())

p1 13.0
p2 3.0
p3 7.0
p4 -0.0
obj= 79.0
